## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.20.0


## Load Dataset

In [2]:
df = pd.read_csv('bengaluru_house_prices.csv')

df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


## Data Preprocessing & Cleaning

In [3]:
df = df.drop(columns=['area_type', 'availability', 'society', 'balcony'])

df = df.dropna()

df['bhk'] = df['size'].apply(lambda x: int(x.split(' ')[0]))
df = df.drop(columns=['size'])

def convert_sqft_to_num(x):
    tokens = str(x).split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(x)
    except:
        return None

df['total_sqft'] = df['total_sqft'].apply(convert_sqft_to_num)
df = df.dropna()

location_stats = df['location'].value_counts()
locations_less_than_10 = location_stats[location_stats <= 10]
df['location'] = df['location'].apply(lambda x: 'other' if x in locations_less_than_10 else x)

df = pd.get_dummies(df, columns=['location'], drop_first=True, dtype=int)

df.head()

,total_sqft,bath,price,bhk,location_1st Block Jayanagar,location_1st Phase JP Nagar,location_2nd Phase Judicial Layout,location_2nd Stage Nagarbhavi,location_5th Block Hbr Layout,location_5th Phase JP Nagar,...,location_Vishveshwarya Layout,location_Vishwapriya Layout,location_Vittasandra,location_Whitefield,location_Yelachenahalli,location_Yelahanka,location_Yelahanka New Town,location_Yelenahalli,location_Yeshwanthpur,location_other
0,1056.0,2.0,39.07,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2600.0,5.0,120.00,4,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1440.0,2.0,62.00,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1521.0,3.0,95.00,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1200.0,2.0,51.00,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Split Data & Feature Scaling

In [4]:
X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train_scaled.shape}")

X_train shape: (10560, 243)


## Build ANN Model

In [5]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        31,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,601 (162.50 KB)

 Trainable params: 41,601 (162.50 KB)

 Non-trainable params: 0 (0.00 B)

## Train the Model

In [6]:
history = model.fit(X_train_scaled, y_train,
                    validation_split=0.2,
                    epochs=60,
                    batch_size=32,
                    verbose=1)

Epoch 1/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 20488.9375 - mae: 67.4978 - val_loss: 18105.9590 - val_mae: 55.5558
Epoch 2/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 14062.2129 - mae: 48.1749 - val_loss: 17381.7500 - val_mae: 43.8582
Epoch 3/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 12974.8975 - mae: 45.0209 - val_loss: 16336.7598 - val_mae: 43.4360
Epoch 4/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 12877.5479 - mae: 44.4407 - val_loss: 17313.9219 - val_mae: 46.0210
Epoch 5/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 12191.4678 - mae: 43.2102 - val_loss: 15861.9502 - val_mae: 47.6475
Epoch 6/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 12344.6299 - mae: 43.2655 - val_loss: 15597.4756 - val_mae: 44.4768
Epoch 7/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 12279.6943 - mae: 43.8684 - val_loss: 15298.5898 - val_mae: 40.4642
Epoch 8/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 11738.7891 - mae: 42.7369 - val_loss: 15229.

## Evaluate the Model

In [7]:
mse, mae = model.evaluate(X_test_scaled, y_test)
print(f"Test Mean Absolute Error (MAE): {mae:.2f}")

83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 7754.2998 - mae: 35.5727
Test Mean Absolute Error (MAE): 35.57
